In [1]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [2]:
print("Loading dataset...")
data = pd.read_csv(
    "Downloads/archive (2)/training.1600000.processed.noemoticon.csv",
    encoding='latin-1',
    header=None,
    usecols=[0,1,2,3,4,5]
)
data.head()

Loading dataset...


FileNotFoundError: [Errno 2] No such file or directory: 'Downloads/archive (2)/training.1600000.processed.noemoticon.csv'

In [52]:
# Step 2: Rename (ONLY ONCE)
data.columns = ["target", "id", "date", "flag", "user", "text"]

# Step 3: Take sample
data = data.sample(5000, random_state=42)

print(data.shape)
print(data.columns)

(5000, 6)
Index(['target', 'id', 'date', 'flag', 'user', 'text'], dtype='object')


In [53]:
def extract_hashtags(text):
    return re.findall(r"#(\w+)", str(text))

data["hashtags"] = data["text"].apply(extract_hashtags)

data["label"] = data["hashtags"].apply(
    lambda x: x[0] if len(x) > 0 else np.random.choice(["AI","sports","music"])
)

In [54]:
texts = data["text"].astype(str).values
labels = data["label"].values

tokenizer = Tokenizer(num_words=2000)
tokenizer.fit_on_texts(texts)

sequences = tokenizer.texts_to_sequences(texts)
X = pad_sequences(sequences, maxlen=15)

# Encode labels
encoder = LabelEncoder()
y = encoder.fit_transform(labels)

In [55]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [56]:
model = Sequential()
model.add(Embedding(input_dim=2000, output_dim=64))
model.add(LSTM(64))
model.add(Dense(len(set(y)), activation='softmax'))

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)              │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_3 (LSTM)                        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
print("\nTraining model...")
history = model.fit(
    X_train, y_train,
    epochs=3,
    batch_size=32,
    validation_data=(X_test, y_test)
)


Training model...
Epoch 1/3
125/125 ━━━━━━━━━━━━━━━━━━━━ 12s 37ms/step - accuracy: 0.3237 - loss: 1.8007 - val_accuracy: 0.3180 - val_loss: 1.3347
Epoch 2/3
125/125 ━━━━━━━━━━━━━━━━━━━━ 4s 26ms/step - accuracy: 0.3358 - loss: 1.2760 - val_accuracy: 0.3330 - val_loss: 1.3309
Epoch 3/3
118/125 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.3337 - loss: 1.2824

In [ ]:
loss, acc = model.evaluate(X_test, y_test)
print("\nModel Accuracy:", acc)

In [ ]:
def predict_trend(text):
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=15)
    pred = model.predict(padded, verbose=0)
    label = encoder.inverse_transform([np.argmax(pred)])
    return label[0]

In [ ]:
# Example prediction
sample = "football match today amazing goal"
print("\nSample Text:", sample)
print("Predicted Trend:", predict_trend(sample))

In [ ]:
all_tags = sum(data["hashtags"], [])
tag_counts = pd.Series(all_tags).value_counts().head(10)

print("\nTop Trending Hashtags:")
print(tag_counts)

In [ ]:
# Plot graph
tag_counts.plot(kind='bar')
plt.title("Top Trending Hashtags")
plt.xlabel("Hashtags")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
print(data.shape)
print(data.columns)

In [ ]:
print(data.columns)

In [ ]:
import time

print("\nReal-Time Trend Simulation:\n")

for i in range(5):
    text = data["text"].iloc[i]
    print("Tweet:", text[:50])
    print("Trend:", predict_trend(text))
    print("-"*40)
    time.sleep(1)